### DermAI Diagnostics — Predictive Modelling



- Logistic Regression
- Random Forest 
- XGBoost

- Performance is compared using:

- Accuracy
-         
- Recall
- F1-score
- ROC-AUC                   



### STEP 1 - Import Relevant Libraries

In [ ]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn
import seaborn as sns
 

warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier



: 

In [ ]:
# Set Project Paths

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "Models")
REPORTS_DIR = os.path.join("..", "reports")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

### STEP 2 — Load Training and Testing Data

In [ ]:

X_train = pd.read_csv(
    os.path.join(PROCESSED_DIR, "X_train.csv")
)

X_test = pd.read_csv(
    os.path.join(PROCESSED_DIR, "X_test.csv")
)

y_train = (
    pd.read_csv(
        os.path.join(PROCESSED_DIR, "y_train.csv")
    )
    .squeeze("columns")
)

y_test = (
    pd.read_csv(
        os.path.join(PROCESSED_DIR, "y_test.csv")
    )
    .squeeze("columns")
)

In [ ]:
# step 2b: check shape
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)




In [ ]:
# step 2c: Load the Preprocessor
preprocessor = joblib.load(
    os.path.join(MODELS_DIR, "preprocessor.joblib")
)


In [ ]:
# step 2d: Check Class Balance
class_counts = y_train.value_counts().sort_index()

print(class_counts)

### STEP 3 - Logistic Regression

In [ ]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [ ]:
# Train Logistic Regression

lr_model.fit(X_train,y_train)

In [ ]:
# Logistic Regression Predictions
lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

# Logistic Regression Evaluation
lr_results = pd.DataFrame({

    "Model": ["Logistic Regression"],

    "Accuracy": [
        accuracy_score(y_test, lr_pred)
    ],

    "Precision": [
        precision_score(y_test, lr_pred)
    ],

    "Recall": [
        recall_score(y_test, lr_pred)
    ],

    "F1 Score": [
        f1_score(y_test, lr_pred)
    ],

    "ROC AUC": [
        roc_auc_score(y_test, lr_prob)
    ]

})

lr_results.round(3)

In [ ]:
# Step 3e — Classification Report
print(
    classification_report(
        y_test,
        lr_pred,
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)

### STEP 4 — Random Forest Classifier

- Non-linear relationships
- Feature interactions
- Complex decision boundaries


In [ ]:
# step 4a — Build and Train the Model
rf_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])


In [ ]:
# step 4b:  the Model
rf_model.fit(
    X_train,
    y_train
)

In [ ]:
# Step 4c — Make Predictions
y_pred_rf = rf_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
# Step 4d — Evaluate Random Forest
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred_rf, zero_division=0))
print("ROC AUC  :", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report")

print(
    classification_report(    
        y_test,
        y_pred_rf,  
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)



In [ ]:
# Step 4e — Confusion Matrix
cm_rf = confusion_matrix(
    y_test,
    y_pred_rf
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["Benign", "Malignant"]
).plot(cmap="Blues")

plt.title("Random Forest Confusion Matrix")
plt.show()


### STEP 5 — XGBoost Classifier


In [ ]:
# step 5a: calculate the class weight from the training target.

class_counts = y_train.value_counts().sort_index()

negative_count = class_counts.get(0, 0)
positive_count = class_counts.get(1, 0)

scale_pos_weight = negative_count / positive_count

print("Benign training cases   :", negative_count)
print("Malignant training cases:", positive_count)
print("Scale pos weight        :", round(scale_pos_weight, 3))

In [ ]:
# step 5b: Build the Model
xgb_model = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier", XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",     
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# step 5c: Train XGBoost
xgb_model.fit(
    X_train,
    y_train
)

In [ ]:
# step 5d: Make Predictions
y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# Step 4d — Evaluate Xgboost
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred_xgb, zero_division=0))
print("ROC AUC  :", roc_auc_score(y_test, y_prob_xgb))

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)



In [ ]:
# step 5f: Confusion Matrix
cm_xgb = confusion_matrix(
    y_test,
    y_pred_xgb
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=["Benign", "Malignant"]
).plot(cmap="Blues")

plt.title("XGBoost Confusion Matrix")
plt.show()

### STEP 6 - Hyperparameter Tuning

In [ ]:
# Hyperparameter Tuning

# The initial comparison showed that Logistic Regression achieved the highest performance. However, tree-based models often benefit from hyperparameter tuning.

# In this step, RandomizedSearchCV is used to optimise:

# Random Forest
# XGBoost

# The models are evaluated using **ROC AUC**, which is an appropriate metric for this healthcare classification problem because it measures the model's ability to distinguish between benign and malignant lesions across different classification thresholds.

# RandomizedSearchCV is selected instead of GridSearchCV because it explores a wide range of parameter combinations while significantly reducing computational time.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

models = {
    "Random Forest": (
        rf_model,
        {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],                 
            "classifier__min_samples_leaf": [1, 2]
        }
    ),

    "XGBoost": (
        xgb_model,
        {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__learning_rate": [0.01, 0.05, 0.1],
            "classifier__max_depth": [3, 5, 7]
        }
    )
}

best_models = {}

for name, (pipeline, params) in models.items():

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,
        cv=5,
        scoring="roc_auc",
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_

    print(f"\n{name}")
    print("Best Parameters:")
    print(search.best_params_)
    print(f"Best CV ROC AUC: {search.best_score_:.3f}")

In [ ]:
# step 6b -  evaluate tuned models

tuned_results = []

for name, model in best_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    tuned_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "ROC AUC": roc_auc_score(
            y_test,
            y_prob
        )
    })

tuned_results = pd.DataFrame(tuned_results)

tuned_results.round(3)

In [ ]:
# step 6c - Combine Logistic Regression with tuned models

final_comparison = pd.concat(

    [lr_results, tuned_results],

    ignore_index=True

)

final_comparison.round(3)

In [ ]:
# step 6d: Rnak the models
model_ranking = final_comparison.sort_values(
    by="ROC AUC",
    ascending=False
).reset_index(drop=True)

model_ranking.round(3)

In [ ]:
metrics = [

    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"

]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15,8)
)

axes = axes.flatten()

for ax, metric in zip(axes, metrics):

    sns.barplot(

        data=model_ranking,

        x="Model",

        y=metric,

        ax=ax

    )

    ax.set_title(metric)

    ax.tick_params(
        axis="x",
        rotation=15
    )

plt.delaxes(axes[-1])

plt.tight_layout()

plt.show()

In [ ]:
# Step 6e — Identify the Initial Best Model
best_model_name = final_comparison.iloc[0]["Model"]

print(f"Best Model: {best_model_name}")

In [ ]:
### Model Comparison Summary

# final comparison shows the performance of all candidate models on the independent test dataset.

#  model with the highest ROC AUC was selected as the final model because it demonstrated the best ability to distinguish between benign and malignant lesions while maintaining strong recall and F1 score.

# selected model or feature importance analysis, explainability, API development, and deployment.

In [ ]:
# step 6f - select best model
model_lookup = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

best_model = model_lookup[best_model_name]

print("Selected model:", best_model_name)

### STEP 7 — SHAP Explainability

- Machine learning models often achieve high predictive performance but can be difficult to interpret.

- SHAP (SHapley Additive exPlanations) explains how each feature contributes to an individual prediction and the model as a whole.

- This improves transparency and supports trust in AI-assisted clinical decision-making.

In [ ]:
# step 7a - Transform the test data
import shap

X_test_processed = best_model.named_steps["preprocessor"].transform(X_test)

# Get the trained Logistic Regression model
classifier = best_model.named_steps["classifier"]

# step 7b - Create SHAP explainer
explainer = shap.LinearExplainer(
    classifier,
    X_test_processed
)

# step 7c -  Calculate SHAP values
shap_values = explainer(X_test_processed)

In [ ]:
# step 7d - get feature names
feature_names = (
    best_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

In [ ]:
# step 7e - summary plot
shap.summary_plot(
    shap_values.values,
    X_test_processed,
    feature_names=feature_names,
    plot_type="bar"
    
)

### STEP 8 - Select and Save the best model

In [ ]:
# step 8a - Select the trained model

if best_model_name == "Logistic Regression":
    best_model = lr_model

elif best_model_name == "Random Forest":
    best_model = best_models["Random Forest"]

else:
    best_model = best_models["XGBoost"]

In [ ]:
# step 8b - create the path

MODELS_DIR = os.path.join(
    "..",
    "Models"
)

REPORTS_DIR = os.path.join(
    "..",
    "reports"
)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

In [ ]:
# step 8c - save models

final_comparison.to_csv(
    os.path.join(
        REPORTS_DIR,
        "final_comparison.csv"
    ),
    index=False
)

joblib.dump(
    lr_model,
    os.path.join(
        MODELS_DIR,
        "logistic_regression_pipeline.joblib"
    )
)

joblib.dump(
    rf_model,
    os.path.join(
        MODELS_DIR,
        "random_forest_pipeline.joblib"
    )
)

joblib.dump(
    xgb_model,
    os.path.join(
        MODELS_DIR,
        "xgboost_pipeline.joblib"
    )
)

joblib.dump(
    best_model,
    os.path.join(
        MODELS_DIR,
        "best_model_pipeline.joblib"
    )
)

print("Models and comparison results saved successfully.")

In [ ]:
# step 8d - Feature Importance


fitted_preprocessor = (
    lr_model
    .named_steps["preprocessor"]
)

feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

xgb_importances = (
    xgb_model
    .named_steps["classifier"]
    .feature_importances_
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_importances
})

In [ ]:
feature_importance["Feature"] = (
    feature_importance["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

In [ ]:
top20 = (
    feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
    .head(20)
)

top20

In [ ]:
plt.figure(figsize=(10, 8))

plt.barh(
    top20["Feature"][::-1],
    top20["Importance"][::-1]
)

plt.title(
    "Top 20 Most Important Features — Logistic  Regression",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# save the feature importance 
feature_importance.to_csv(
    os.path.join(
        REPORTS_DIR,
        "lr_feature_importance.csv"
    ),
    index=False
)